# H&M 資料探索：建模共識的推導

這份 notebook 的終點是最後一格的**建模共識**——後續召回、排序、對話層都照那份規格做。

前面每一節都在回答同一個問題：**共識裡那一條，是怎麼推出來的？**
章節與共識的四節一一對應：

| 本 notebook | 推導出共識的 |
|---|---|
| §1 附錄 A 的 19 項統計 | 資料的基本輪廓（共識引用為「資料本身長什麼樣」） |
| §2 時間切分與評估母體 | 共識 §1、§2 |
| §3 缺失值 | 共識 §3 |
| §4 基準線 | 共識 §4 |
| §5 與會議記錄比對 | 共識 §5 |

**收錄標準**：說得出它推導出共識的哪一條。說不出來的就不在這裡。

**執行前提**：先跑過 `python main.py build` 把 CSV 轉成 Parquet。全部執行約 15 秒。

**效能原則**：所有聚合都在 SQL 裡完成，只把小結果交給 Python。不把 3178 萬筆明細拉進記憶體。

In [ ]:
import duckdb                    # appendix_a 的型別標註需要
import matplotlib.pyplot as plt
import pandas as pd

import dataset
from dataset import LAST_DAY, HOLDOUT_START, LAST_4W_START, PRICE_SCALE

# macOS 的中文字型，否則圖上的中文會變成方框
plt.rcParams["font.sans-serif"] = ["PingFang TC"]
plt.rcParams["axes.unicode_minus"] = False

con = dataset.connect()
print("已連線。交易筆數：", f"{con.execute('SELECT count(*) FROM tx').fetchone()[0]:,}")

---

## 1. 附錄 A 的 19 項統計

與提案附錄 A 逐字一致。共識開頭寫的「資料本身長什麼樣請看 §1」指的就是這裡。
改動任何一行都會讓表格與提案不一致，要改請連同提案一起更新。

In [ ]:
def appendix_a(con: duckdb.DuckDBPyConnection) -> list[tuple[str, str]]:
    """重算提案附錄 A 的每一列，回傳 (項目, 數值) 的清單。"""
    q = lambda sql: con.execute(sql).fetchone()
    rows: list[tuple[str, str]] = []

    # 交易期間與總量
    mn, mx, n_tx, n_cust_tx, n_art_sold = q("""
        SELECT min(t_dat), max(t_dat), count(*),
               count(DISTINCT customer_id), count(DISTINCT article_id) FROM tx
    """)
    n_cust_all = q("SELECT count(*) FROM cus")[0]
    n_art_all = q("SELECT count(*) FROM art")[0]
    rows.append(("交易期間", f"{mn} 至 {mx}"))
    rows.append(("交易筆數", f"{n_tx:,}"))
    rows.append(("有交易的顧客數 / 顧客總數", f"{n_cust_tx:,} / {n_cust_all:,}"))
    rows.append(("有成交的品項數 / 品項總數", f"{n_art_sold:,} / {n_art_all:,}"))

    # 最近四週有成交的品項
    n_4w = q(f"SELECT count(DISTINCT article_id) FROM tx WHERE t_dat >= '{LAST_4W_START}'")[0]
    rows.append(("最近四週有成交的品項", f"{n_4w:,}（{n_4w / n_art_all:.1%}）"))

    # 通路占比。通路 2 在 COVID 封城期間佔 100%，這是判定 2 = 線上的依據。
    c1, c2 = q("""
        SELECT round(100.0 * count(*) FILTER (WHERE sales_channel_id = 1) / count(*), 1),
               round(100.0 * count(*) FILTER (WHERE sales_channel_id = 2) / count(*), 1) FROM tx
    """)
    rows.append(("通路 1（實體店）/ 通路 2（線上）交易占比", f"{c1}% / {c2}%"))

    a1, a2 = q("""
        SELECT round(100.0 * count(*) FILTER (WHERE sales_channel_id = 1) / count(*), 1),
               round(100.0 * count(*) FILTER (WHERE sales_channel_id = 2) / count(*), 1)
        FROM tx WHERE t_dat >= '2020-04-01' AND t_dat < '2020-05-01'
    """)
    rows.append(("2020 年 4 月通路 1（實體店）/ 通路 2（線上）占比（COVID 封城，門市關閉）",
                 f"{a1}% / {a2}%"))

    # 單日購物籃平均件數：線上訂單通常一次買比較多件，是通路判定的輔助證據。
    b1, b2 = q("""
        SELECT round(avg(n) FILTER (WHERE sales_channel_id = 1), 1),
               round(avg(n) FILTER (WHERE sales_channel_id = 2), 1)
        FROM (SELECT customer_id, t_dat, sales_channel_id, count(*) AS n
              FROM tx GROUP BY 1, 2, 3)
    """)
    rows.append(("單日購物籃平均件數：通路 1（實體店）/ 通路 2（線上）", f"{b1} / {b2}"))

    # 價格 ×590 還原後最常見的價位，用來佐證那個倍率是對的
    top_prices = con.execute(f"""
        SELECT round(price * {PRICE_SCALE}, 2) AS p FROM tx
        GROUP BY 1 ORDER BY count(*) DESC LIMIT 3
    """).fetchall()
    rows.append((f"價格 ×{PRICE_SCALE} 後最常見的三個價位",
                 "、".join(f"{p[0]:g}" for p in top_prices)))

    p10, p50, p90, pmax = q("""
        SELECT round(quantile_cont(price, 0.1), 4), round(quantile_cont(price, 0.5), 4),
               round(quantile_cont(price, 0.9), 4), round(max(price), 4) FROM tx
    """)
    rows.append(("價格 P10 / P50 / P90 / 最大", f"{p10} / {p50} / {p90} / {pmax}"))

    # 每位顧客的交易筆數分布
    med, mean, cp90, once = q("""
        SELECT quantile_cont(n, 0.5), round(avg(n), 1), quantile_cont(n, 0.9),
               count(*) FILTER (WHERE n = 1)
        FROM (SELECT customer_id, count(*) AS n FROM tx GROUP BY 1)
    """)
    rows.append(("每位顧客交易筆數中位數 / 平均 / P90", f"{med:g} / {mean} / {cp90:g}"))
    rows.append(("只買過一次的顧客", f"{once:,}（{once / n_cust_tx:.1%}）"))

    # 同品項回購：rn > 1 代表這位顧客先前買過同一件
    repeat = q("""
        SELECT round(100.0 * count(*) FILTER (WHERE rn > 1) / count(*), 1)
        FROM (SELECT row_number() OVER (PARTITION BY customer_id, article_id ORDER BY t_dat) AS rn
              FROM tx)
    """)[0]
    rows.append(("同品項回購占交易比例", f"{repeat}%"))

    # 長尾集中度：前 1% 與前 10% 的品項各佔多少交易
    t1, t10 = q("""
        WITH s AS (SELECT article_id, count(*) AS n FROM tx GROUP BY 1),
             r AS (SELECT n, row_number() OVER (ORDER BY n DESC) AS rk, count(*) OVER () AS tot FROM s)
        SELECT round(100.0 * sum(n) FILTER (WHERE rk <= tot * 0.01) / sum(n), 1),
               round(100.0 * sum(n) FILTER (WHERE rk <= tot * 0.10) / sum(n), 1) FROM r
    """)
    rows.append(("前 1% / 前 10% 品項的交易占比", f"{t1}% / {t10}%"))

    # 保留週（最後 7 天），也就是離線評估要預測的那一週
    h_tx, h_cust, h_art = q(f"""
        SELECT count(*), count(DISTINCT customer_id), count(DISTINCT article_id)
        FROM tx WHERE t_dat >= '{HOLDOUT_START}'
    """)
    rows.append((f"保留週（{HOLDOUT_START[5:]} 至 {LAST_DAY[5:]}）交易 / 顧客 / 品項",
                 f"{h_tx:,} / {h_cust:,} / {h_art:,}"))

    # 保留週買家裡有多少人在那之前就買過東西。剩下的就是冷啟動新客。
    with_hist = q(f"""
        SELECT round(100.0 * count(*) FILTER (WHERE n_prior > 0) / count(*), 1)
        FROM (SELECT customer_id,
                     count(*) FILTER (WHERE t_dat < '{HOLDOUT_START}') AS n_prior,
                     count(*) FILTER (WHERE t_dat >= '{HOLDOUT_START}') AS n_last
              FROM tx GROUP BY 1) WHERE n_last > 0
    """)[0]
    rows.append(("保留週顧客有歷史交易的比例", f"{with_hist}%"))

    # 品牌線與商品群組的組成
    idx = con.execute("""
        SELECT a.index_group_name, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
        FROM tx t JOIN art a USING (article_id) GROUP BY 1 ORDER BY 2 DESC
    """).fetchall()
    rows.append(("品牌線占比", "、".join(f"{n} {p}%" for n, p in idx)))

    grp = con.execute("""
        SELECT a.product_group_name, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
        FROM tx t JOIN art a USING (article_id) GROUP BY 1 ORDER BY count(*) DESC LIMIT 4
    """).fetchall()
    rows.append(("商品群組占比（前四）", "、".join(f"{n} {p}%" for n, p in grp)))

    # 2020 年的季節性
    monthly = con.execute("""
        SELECT strftime(t_dat, '%m') AS m, count(*) AS n
        FROM tx WHERE t_dat >= '2020-01-01' GROUP BY 1 ORDER BY 1
    """).fetchall()
    rows.append(("2020 年月交易量",
                 "、".join(f"{int(m)} 月{'（至 22 日）' if m == '09' else ''} {n / 10000:.0f} 萬"
                            for m, n in monthly)))

    return rows


# --------------------------------------------------------------------------

def print_markdown(rows: list[tuple[str, str]]) -> None:
    """印成 Markdown 表格，可直接貼回提案附錄 A。"""
    print("\n| 項目 | 數值 |")
    print("|---|---|")
    for item, value in rows:
        print(f"| {item} | {value} |")

執行後的輸出可以直接複製，貼回提案的附錄 A。

In [ ]:
rows = appendix_a(con)
print_markdown(rows)

---

## 2. 時間切分與評估母體

> **推導共識 §1（時間切分）與 §2（評估規格）。**

要定三件事：切在哪一天、母體是誰、正解怎麼算。下面把每個數字算出來，
並量化「切錯一天」的後果。

In [ ]:
print("① 評估母體與正解規模")
print(con.execute(f"""
    SELECT count(DISTINCT customer_id) AS 保留週買家,
           count(*) AS 保留週交易,
           count(DISTINCT article_id) AS 保留週成交品項
    FROM tx WHERE t_dat >= '{HOLDOUT_START}'
""").df().to_string(index=False))

n_pairs = con.execute(f"""
    SELECT count(*) FROM (SELECT DISTINCT customer_id, article_id
                          FROM tx WHERE t_dat >= '{HOLDOUT_START}')
""").fetchone()[0]
print(f"\n正解組數（顧客 × 相異品項）：{n_pairs:,}")

# 會議記錄把切分寫成「9/15–9/22」（8 天）。9/15 放哪一邊，母體就跟著變。
print("\n② 切分邊界：9/15 放錯邊的後果")
print(con.execute(f"""
    SELECT count(*) FILTER (WHERE t_dat = '2020-09-15') AS 九月15日交易,
           count(*) FILTER (WHERE t_dat >= '{HOLDOUT_START}') AS 保留週_09_16起,
           count(*) FILTER (WHERE t_dat >= '2020-09-15') AS 若把09_15也算進去,
           count(DISTINCT customer_id) FILTER (WHERE t_dat >= '{HOLDOUT_START}') AS 買家_09_16起,
           count(DISTINCT customer_id) FILTER (WHERE t_dat >= '2020-09-15') AS 買家_若含09_15
    FROM tx
""").df().to_string(index=False))

# 分母換一個，同一個模型的分數就差一個數量級
print("\n③ 分母的量級差異")
n_buyer = con.execute(f"SELECT count(DISTINCT customer_id) FROM tx WHERE t_dat >= '{HOLDOUT_START}'").fetchone()[0]
n_all = con.execute("SELECT count(*) FROM cus").fetchone()[0]
print(f"   買家分母 {n_buyer:,}   全體顧客分母 {n_all:,}   相差 {n_all/n_buyer:.1f} 倍")

# 冷啟動比例：決定要不要另外處理沒有歷史的人
print("\n④ 保留週買家有多少人在那之前買過東西")
print(con.execute(f"""
    SELECT round(100.0*count(*) FILTER (WHERE n_prior > 0)/count(*), 1) AS 有歷史百分比,
           count(*) FILTER (WHERE n_prior = 0) AS 冷啟動人數
    FROM (SELECT customer_id,
                 count(*) FILTER (WHERE t_dat <  '{HOLDOUT_START}') AS n_prior,
                 count(*) FILTER (WHERE t_dat >= '{HOLDOUT_START}') AS n_last
          FROM tx GROUP BY 1) WHERE n_last > 0
""").df().to_string(index=False))

**推導出共識 §1 與 §2 的四條**：

1. **母體 = 68,984 位保留週買家、正解 213,728 組**（① 的輸出）
2. **切分必須是 `t_dat < '2020-09-16'` / 保留週 09-16 ~ 09-22**。9/15 那天有 26,053 筆交易，
   放進保留週的話母體會變，與提案和彼此都不可比（② 的輸出）
3. **分母一定要註明**。買家分母與全體顧客分母差 19.9 倍，同一個模型會報出兩個差一個數量級的分數（③）
4. **冷啟動 8.1% 計入主分數**，不另外排除——排除會讓分數虛高（④）

還有一條是規格而非數字：**驗證週（09-09 ~ 09-15）用來調參數，保留週只在最後跑一次。**
理由在 §4.2 會用實際數字說明。

---

## 3. 缺失值

> **推導共識 §3 的整張規格表。**

會議當時的規則是「缺失多的補、缺失少的刪」。下面先看缺失狀況，
再量化「刪」到底刪掉什麼——因為刪維度表（`cus` / `art`）的列，跟刪事實表（`tx`）的列後果完全不同。

In [ ]:
for t in ("tx", "cus", "art"):
    cols = list(con.execute(f"DESCRIBE {t}").df()["column_name"])
    sel = ", ".join(f'round(100.0*count(*) FILTER (WHERE "{c}" IS NULL)/count(*),2) AS "{c}"' for c in cols)
    r = con.execute(f"SELECT {sel} FROM {t}").df().T
    r.columns = ["缺失%"]
    bad = r[r["缺失%"] > 0]
    print(f"\n{t}（{len(cols)} 欄）")
    if bad.empty:
        print("   全部無缺失")
    for c, v in bad.itertuples():
        n = con.execute(f'SELECT count(*) FROM {t} WHERE "{c}" IS NULL').fetchone()[0]
        print(f"   {c:<26} {v:>5.2f}%   {n:>7,} 列")

In [ ]:
# 「刪除缺失列」的真正代價：看它會不會動到評估母體與召回上限
print("刪除 age 缺失的顧客")
print(con.execute(f"""
    SELECT count(DISTINCT c.customer_id) AS 顧客數,
           count(t.customer_id) AS 交易筆數,
           count(*) FILTER (WHERE t.t_dat >= '{HOLDOUT_START}') AS 保留週交易,
           count(DISTINCT t.customer_id) FILTER (WHERE t.t_dat >= '{HOLDOUT_START}') AS 保留週買家
    FROM cus c LEFT JOIN tx t USING (customer_id) WHERE c.age IS NULL
""").df().to_string(index=False))

print("\n刪除 detail_desc 缺失的商品")
print(con.execute(f"""
    SELECT count(DISTINCT a.article_id) AS 商品數,
           count(t.article_id) AS 交易筆數,
           count(DISTINCT t.article_id) FILTER (WHERE t.t_dat >= '{HOLDOUT_START}') AS 保留週正解品項
    FROM art a LEFT JOIN tx t USING (article_id) WHERE a.detail_desc IS NULL
""").df().to_string(index=False))

# FN / Active 補 0 的依據：這兩欄從來沒有出現過 0
print("\nFN / Active 實際出現過哪些值")
print(con.execute("""
    SELECT count(DISTINCT FN) AS FN相異值, min(FN) AS FN最小, max(FN) AS FN最大,
           count(DISTINCT Active) AS Active相異值, min(Active) AS Active最小, max(Active) AS Active最大
    FROM cus
""").df().to_string(index=False))

**推導出共識 §3 的每一格**：

- **`age` 缺 1.16% 但含 292 位保留週買家** → 刪掉母體從 68,984 變 68,692，與共識 §2 定的評估母體衝突。
  **保留為 `未填` 類別**
- **`detail_desc` 缺 0.39% 但含 17 個保留週正解品項** → 刪掉等於自己降低召回上限。**補空字串不刪商品**
- **`FN` / `Active` 只出現過 `1.0`，資料裡沒有任何 `0`** → NULL 的語意就是「沒有這個標記」，**補 0** 是還原原意
- 因此判準不是缺失比例，而是**看 NULL 的語意 + 看刪的是哪張表**

---

## 4. 基準線

> **推導共識 §4。**

要交付 B0（全站熱門）與 B1（分群熱門）。三個決定要推：**分群怎麼切**、
**熱銷區間取幾天**、**結果能不能重現**。

### 4.1 分群依據：年齡怎麼分箱

B1 的分群是「年齡層 × 品牌線」。年齡要分成幾段，看分布決定——
每段要有足夠的人數，分群熱銷榜才有統計意義。

In [ ]:
age = con.execute("""
    SELECT CASE WHEN age IS NULL THEN '未填' WHEN age < 20 THEN '<20'
                WHEN age < 30 THEN '20-29' WHEN age < 40 THEN '30-39'
                WHEN age < 50 THEN '40-49' WHEN age < 60 THEN '50-59'
                ELSE '60+' END AS band,
           count(*) AS n
    FROM cus GROUP BY 1
""").df()

order = ["<20", "20-29", "30-39", "40-49", "50-59", "60+", "未填"]
age = age.set_index("band").reindex(order).reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(age["band"], age["n"] / 1e4, color="#4C72B0")
ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=9)
ax.set_title("顧客年齡分布（共 137 萬人）")
ax.set_ylabel("人數（萬）")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

age

十歲一段，每段都在 10 萬人以上，最小的 `<20` 也有 6 萬多。
`未填` 自成一類（§3 已定不刪），所以是 **7 個年齡層**。

In [ ]:
import numpy as np
import datetime as dt

FIRST_DAY = "2018-09-20"
VAL_START = "2020-09-09"     # 驗證週：調參只能看這裡
VAL_END   = "2020-09-16"
HOLD_END  = "2020-09-23"


def ap_at_k(truth: set, pred: list, k: int = 12) -> float:
    """單一顧客的 Average Precision@K，分母 min(正解數, K)，與 Kaggle 一致。

    先去重：同一件商品在推薦清單裡出現兩次，會被算成兩次命中，
    分數可能超過 1（實測 [A,A,B] 對正解 {A,B} 會得到 1.5）。
    規格規定清單不得重複，這裡再擋一次，讓錯誤的清單也算出對的分數。
    """
    pred = list(dict.fromkeys(pred))   # 去重且保留順序
    hits = 0
    score = 0.0
    for i, a in enumerate(pred[:k], 1):
        if a in truth:
            hits += 1
            score += hits / i
    return score / min(len(truth), k)


def top_k(start: str, end: str, k: int = 12) -> list[str]:
    """[start, end) 內成交最多的 k 個品項。同票以 article_id 決勝，確保可重現。"""
    return [r[0] for r in con.execute(f"""
        SELECT article_id FROM tx WHERE t_dat >= '{start}' AND t_dat < '{end}'
        GROUP BY 1 ORDER BY count(*) DESC, article_id LIMIT {k}
    """).fetchall()]


def truth_of(start: str, end: str) -> list[list[str]]:
    """[start, end) 內每位買家買過的相異品項。"""
    return list(con.execute(f"""
        SELECT list(DISTINCT article_id) FROM tx
        WHERE t_dat >= '{start}' AND t_dat < '{end}' GROUP BY customer_id
    """).df().iloc[:, 0])

### 4.2 熱銷區間取幾天：用驗證週決定，不看保留週

候選有 7 / 14 / 21 / 30 / 60 天與「保留週以外全部」。

**選哪一個是參數決定，而做決定就要看資料——看了哪份資料，那份就不能再拿來誠實評估。**
所以切三份：訓練資料算熱銷榜、**驗證週（09-09 ~ 09-15）決定區間**、保留週只在最後跑一次。

In [ ]:
gt_val = truth_of(VAL_START, VAL_END)
gt_hold = truth_of(HOLDOUT_START, HOLD_END)
gt_val = [set(t) for t in gt_val]
gt_hold = [set(t) for t in gt_hold]

rows = []
for d in (7, 14, 21, 30, 60, "全部"):
    vs = FIRST_DAY if d == "全部" else (dt.date.fromisoformat(VAL_START) - dt.timedelta(days=d)).isoformat()
    hs = FIRST_DAY if d == "全部" else (dt.date.fromisoformat(HOLDOUT_START) - dt.timedelta(days=d)).isoformat()
    # 熱銷榜每個區間只查一次。寫在下面的迴圈裡會變成每位顧客查一次 SQL（84 萬次），跑不完。
    top_val, top_hold = top_k(vs, VAL_START), top_k(hs, HOLDOUT_START)
    rows.append({
        "熱銷區間": "保留週以外全部" if d == "全部" else f"近 {d} 天",
        "驗證週 MAP@12": np.mean([ap_at_k(t, top_val) for t in gt_val]),
        "保留週 MAP@12": np.mean([ap_at_k(t, top_hold) for t in gt_hold]),
    })

sweep = pd.DataFrame(rows)
best = sweep.loc[sweep["驗證週 MAP@12"].idxmax(), "熱銷區間"]
print(f"驗證週買家 {len(gt_val):,}   保留週買家 {len(gt_hold):,}")
print(f"→ 只看驗證週選出來的是：{best}\n")
sweep.round(5)

**結果：驗證週選出 14 天。**

注意兩件事。第一，近 7 天在保留週拿 0.00875 是六個候選裡最高的，但在驗證週只排第二 ——
**那個高分有一部分是那一週的運氣**。第二，14 天在兩週分別是 0.00700 與 0.00710，幾乎一樣；
7 天則是 0.00660 → 0.00875，跳了 33%。**穩定性本身就是可信度的證據。**

`POP_DAYS = 14` 是這樣來的，不是拍腦袋，也不是看保留週挑的。

In [ ]:
POP_DAYS = 14   # 由 4.2 的驗證週掃描決定
POP_START = (dt.date.fromisoformat(HOLDOUT_START) - dt.timedelta(days=POP_DAYS)).isoformat()


def build_segments(train_end: str) -> None:
    """建立 seg 檢視表：每位顧客的 (年齡層, 主要品牌線)。

    年齡缺失保留為「未填」自成一類，不刪顧客（理由見共識 §3）。
    品牌線取訓練期交易數最多的那條，平手時以名稱字母序決勝 ——
    arg_max() 在平手時不具決定性，會讓基準線每次跑出不同結果。
    """
    con.execute(f"""
        CREATE OR REPLACE VIEW seg AS
        SELECT c.customer_id,
               CASE WHEN c.age IS NULL THEN '未填' WHEN c.age < 20 THEN '<20'
                    WHEN c.age < 30 THEN '20-29' WHEN c.age < 40 THEN '30-39'
                    WHEN c.age < 50 THEN '40-49' WHEN c.age < 60 THEN '50-59'
                    ELSE '60+' END AS ageband,
               coalesce(d.ig, '未知') AS ig
        FROM cus c
        LEFT JOIN (
            SELECT customer_id, (list(index_group_name ORDER BY n DESC, index_group_name))[1] AS ig
            FROM (SELECT t.customer_id, a.index_group_name, count(*) AS n
                  FROM tx t JOIN art a USING (article_id)
                  WHERE t.t_dat < '{train_end}' GROUP BY 1, 2)
            GROUP BY 1
        ) d USING (customer_id)
    """)


def evaluate(test_start: str, test_end: str, pop_start: str) -> dict:
    """在 [test_start, test_end) 上評估 B0 與 B1。"""
    b0 = top_k(pop_start, test_start)
    segs = {(r.ageband, r.ig): list(r.top) for r in con.execute(f"""
        SELECT ageband, ig, list(article_id ORDER BY n DESC, article_id)[1:12] AS top
        FROM (SELECT s.ageband, s.ig, t.article_id, count(*) AS n
              FROM tx t JOIN seg s USING (customer_id)
              WHERE t.t_dat >= '{pop_start}' AND t.t_dat < '{test_start}'
              GROUP BY 1, 2, 3)
        GROUP BY 1, 2
    """).df().itertuples()}

    def pad(pred):
        out = list(pred)
        for a in b0:
            if len(out) >= 12:
                break
            if a not in out:
                out.append(a)
        return out[:12]

    gt = con.execute(f"""
        SELECT s.ageband, s.ig, list(DISTINCT t.article_id) AS truth
        FROM tx t JOIN seg s USING (customer_id)
        WHERE t.t_dat >= '{test_start}' AND t.t_dat < '{test_end}'
        GROUP BY t.customer_id, 1, 2
    """).df()
    a0 = np.array([ap_at_k(set(t), b0) for t in gt["truth"]])
    a1 = np.array([ap_at_k(set(r.truth), pad(segs.get((r.ageband, r.ig), []))) for r in gt.itertuples()])
    return {"買家數": len(gt), "分群數": len(segs),
            "B0": a0.mean(), "B1": a1.mean(),
            "B0命中率": (a0 > 0).mean(), "B1命中率": (a1 > 0).mean()}


build_segments(HOLDOUT_START)
r = evaluate(HOLDOUT_START, HOLD_END, POP_START)
print(f"保留週 {HOLDOUT_START} ~ {LAST_DAY}｜母體 {r['買家數']:,} 位買家｜熱銷區間 {POP_DAYS} 天｜{r['分群數']} 個分群\n")
print(f"  B0 全站熱門   MAP@12 = {r['B0']:.5f}   命中率 {r['B0命中率']:.1%}")
print(f"  B1 分群熱門   MAP@12 = {r['B1']:.5f}   命中率 {r['B1命中率']:.1%}   相對 B0 {(r['B1']/r['B0']-1):+.1%}")

### 4.3 可重現性：平手時的決勝

`arg_max()` 與 `ORDER BY count(*) DESC` 在平手時挑哪一個，DuckDB 不保證。
基準線是所有模型要比較的尺，尺本身會抖就沒有意義了。下面是實測。

In [ ]:
# 同一個查詢連跑五次，看分群指派穩不穩定
print("arg_max() 版本（無決勝欄位）")
for k in range(5):
    n = con.execute(f"""
        SELECT count(*) FROM (
          SELECT customer_id, arg_max(index_group_name, n) AS ig FROM (
            SELECT t.customer_id, a.index_group_name, count(*) AS n
            FROM tx t JOIN art a USING (article_id)
            WHERE t.t_dat < '{HOLDOUT_START}' GROUP BY 1, 2) GROUP BY 1)
        WHERE ig = 'Ladieswear'
    """).fetchone()[0]
    print(f"   第 {k+1} 次：Ladieswear 分群 {n:,} 人")

print("\n有多少顧客會平手（最高票不只一條品牌線）")
print(con.execute(f"""
    SELECT count(*) AS 平手顧客數 FROM (
      SELECT customer_id, count(*) FILTER (WHERE n = mx) AS tied FROM (
        SELECT customer_id, index_group_name, n, max(n) OVER (PARTITION BY customer_id) AS mx
        FROM (SELECT t.customer_id, a.index_group_name, count(*) AS n
              FROM tx t JOIN art a USING (article_id)
              WHERE t.t_dat < '{HOLDOUT_START}' GROUP BY 1, 2))
      GROUP BY customer_id) WHERE tied > 1
""").fetchone()[0])

# 加了決勝欄位的版本（build_segments 用的就是這個）連跑五次
print("\n加決勝欄位的版本（list(... ORDER BY n DESC, index_group_name)[1]）")
for k in range(5):
    build_segments(HOLDOUT_START)
    n = con.execute("SELECT count(*) FROM seg WHERE ig = 'Ladieswear'").fetchone()[0]
    print(f"   第 {k+1} 次：Ladieswear 分群 {n:,} 人")

**推導出共識 §4 的三條**：

1. **B0 = 0.00710、B1 = 0.00927（+30.5%）**，熱銷區間 14 天、35 個分群（4.2 的輸出）
2. **14 天是驗證週選出來的，不是看保留週挑的。** 近 7 天在保留週 0.00875 最高但在驗證週只排第二；
   14 天在兩週分別 0.00700 / 0.00710 幾乎一樣——**穩定本身就是可信度的證據**
3. **所有排序都要加決勝欄位。** 74,575 位顧客會平手，`arg_max()` 版本連跑五次分群人數在
   99 萬上下跳動；加了 `ORDER BY n DESC, index_group_name` 之後五次完全相同（4.3 的輸出）

---

## 5. 與 09-23 會議記錄比對

> **推導共識 §5 的對照表。**

會議記錄的每一條都拿資料驗一次。下面先驗要修正的六處，再驗確認無誤的那批。

In [ ]:
print("① 會議理由：『最近 30 天出現的新商品被購買比例最高』")
print(con.execute(f"""
    WITH first AS (SELECT article_id, min(t_dat) AS fd FROM tx GROUP BY 1)
    SELECT CASE WHEN fd >= '2020-08-17' THEN 'A 最近 30 天首次上架'
                WHEN fd >= '2020-07-18' THEN 'B 31-60 天前'
                WHEN fd >= '2020-06-18' THEN 'C 61-90 天前'
                ELSE 'D 更早' END AS 首次上架,
           count(*) AS 商品數, round(avg(n), 1) AS 上架後平均交易數
    FROM first JOIN (SELECT article_id, count(*) AS n FROM tx
                     WHERE t_dat < '{HOLDOUT_START}' GROUP BY 1) USING (article_id)
    GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

print("\n② 會議說『相鄰兩週熱門商品重合率僅約 28%』")
for lab, st in (("2020 全年", "2020-01-01"), ("最近 12 週", "2020-07-01"), ("最近 4 週", "2020-08-19")):
    r = con.execute(f"""
        WITH w AS (SELECT date_trunc('week', t_dat) AS wk, article_id, count(*) AS n
                   FROM tx WHERE t_dat >= '{st}' GROUP BY 1, 2),
             r AS (SELECT wk, article_id,
                          row_number() OVER (PARTITION BY wk ORDER BY n DESC, article_id) AS rk FROM w),
             t AS (SELECT wk, list(article_id) AS top FROM r WHERE rk <= 12 GROUP BY 1)
        SELECT round(avg(len(list_intersect(a.top, b.top)))/12*100, 1), count(*)
        FROM t a JOIN t b ON b.wk = a.wk + INTERVAL 7 DAY
    """).fetchone()
    print(f"   {lab:<10} 重合率 {r[0]:>5.1f}%   （{r[1]} 組相鄰週）")

print("\n③ B1 分群維度：只用年齡 vs 年齡 × 品牌線（熱銷區間同為 14 天）")
gt = con.execute(f"""
    SELECT s.ageband, s.ig, list(DISTINCT t.article_id) AS truth
    FROM tx t JOIN seg s USING (customer_id)
    WHERE t.t_dat >= '{HOLDOUT_START}' GROUP BY t.customer_id, 1, 2
""").df()
b0 = top_k(POP_START, HOLDOUT_START)
def _pad(p):
    out = list(p)
    for a in b0:
        if len(out) >= 12: break
        if a not in out: out.append(a)
    return out[:12]
m_age = {r.ageband: list(r.top) for r in con.execute(f"""
    SELECT ageband, list(article_id ORDER BY n DESC, article_id)[1:12] AS top
    FROM (SELECT s.ageband, t.article_id, count(*) AS n FROM tx t JOIN seg s USING (customer_id)
          WHERE t.t_dat >= '{POP_START}' AND t.t_dat < '{HOLDOUT_START}' GROUP BY 1, 2) GROUP BY 1
""").df().itertuples()}
print(f"   只用年齡（{len(m_age)} 群）    MAP@12 = "
      f"{np.mean([ap_at_k(set(r.truth), _pad(m_age.get(r.ageband, []))) for r in gt.itertuples()]):.5f}")
print(f"   年齡 × 品牌線（35 群）  MAP@12 = 0.00927  （見 §4.2）")

**推導出共識 §5 的六處修正**：

| # | 會議記錄 | 改成 | 這裡的證據 |
|---|---|---|---|
| 1 | 熱銷區間近 30 天 | 近 14 天 | §4.2 掃描表：30 天在驗證週 0.00558、14 天 0.00700 |
| 2 | 理由「最近 30 天新商品被購買比例最高」 | 實測不成立 | ① 最近 30 天上架的商品平均交易數 56.8，比更早上架的 321.9 **低** |
| 3 | B1 只用年齡分群 | 年齡 × 品牌線 | ③ 只用年齡 0.00888（7 群）vs 加品牌線 0.00927（35 群） |
| 4 | `age` 缺失刪除 | 保留為 `未填` | §3：會刪掉 292 位保留週買家 |
| 5 | 缺失 <1–2% 的欄位刪列 | 維度表一律不刪列 | §3：`detail_desc` 那 416 個商品含 17 個保留週正解 |
| 6 | 相鄰兩週重合率 28% | 39–50% | ② 全年 40.1%、最近 12 週 38.9%、最近 4 週 50.0% |

第 7 處（切分寫成 8 天）的證據在 §2 的 ②。

### 5.1 會議記錄中確認無誤的部分

這些照抄進共識，不用改。

In [ ]:
print("① 消費歷史稀疏性：會議說 P50 = 8 個品項")
print(con.execute("""
    SELECT quantile_cont(n_art, 0.5) AS 相異品項中位數, round(avg(n_art), 1) AS 平均,
           quantile_cont(n_tx, 0.5) AS 交易筆數中位數
    FROM (SELECT customer_id, count(DISTINCT article_id) AS n_art, count(*) AS n_tx
          FROM tx GROUP BY 1)
""").df().to_string(index=False))

print("\n② 疫情期間線上通路接近 100%")
print(con.execute("""
    SELECT strftime(t_dat, '%Y-%m') AS 月份,
           count(*) FILTER (WHERE sales_channel_id = 1) AS 通路1_實體,
           count(*) FILTER (WHERE sales_channel_id = 2) AS 通路2_線上
    FROM tx WHERE t_dat >= '2020-02-01' AND t_dat < '2020-07-01'
    GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

print("\n③ 季節性：夏季泳裝")
print(con.execute("""
    SELECT strftime(t.t_dat, '%m') AS 月, count(*) AS 泳裝交易數
    FROM tx t JOIN art a USING (article_id)
    WHERE a.product_type_name IN ('Bikini top', 'Swimwear bottom') AND t.t_dat >= '2020-01-01'
    GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

In [ ]:
monthly = con.execute("""
    SELECT strftime(t_dat, '%Y-%m') AS ym, count(*) AS n
    FROM tx WHERE t_dat >= '2020-01-01'
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly["ym"], monthly["n"] / 1e4, marker="o", color="#C44E52")
ax.axvspan(2.2, 3.5, alpha=0.15, color="gray")
ax.annotate("COVID 封城\n門市關閉", xy=(3.0, monthly["n"].max() / 1e4 * 0.5),
            ha="center", fontsize=9, color="gray")
ax.set_title("2020 年月交易量（9 月僅到 22 日）")
ax.set_ylabel("交易筆數（萬）")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

monthly

**④ 長尾效應**

附錄 A 說「前 1% 品項佔 18.6% 交易、前 10% 佔 60.7%」。這張帕雷托圖畫出完整曲線。

**這是本專案要解決的核心問題**：長尾商品幾乎沒有訓練訊號，也幾乎不會被現有的
「依熱門度排序」介面看見（提案 1.1 節第 2 點）。

In [ ]:
pareto = con.execute("""
    WITH s AS (SELECT article_id, count(*) AS n FROM tx GROUP BY 1),
         r AS (SELECT n,
                      row_number() OVER (ORDER BY n DESC) AS rk,
                      count(*) OVER () AS tot,
                      sum(n) OVER () AS total_n
               FROM s)
    SELECT ceil(100.0 * rk / tot) AS bucket,
           round(100.0 * sum(n) / max(total_n), 4) AS pct
    FROM r GROUP BY 1 ORDER BY 1
""").df()
pareto["cum"] = pareto["pct"].cumsum()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pareto["bucket"], pareto["cum"], color="#4C72B0", linewidth=2)
for x, color in ((1, "#C44E52"), (10, "#DD8452")):
    y = pareto.loc[pareto["bucket"] <= x, "pct"].sum()
    ax.plot([0, x, x], [y, y, 0], linestyle="--", color=color, linewidth=1)
    ax.annotate(f"前 {x}% 品項 → {y:.1f}% 交易",
                xy=(x, y), xytext=(x + 6, y - 6), color=color, fontsize=10)
ax.set_title("商品銷量的長尾集中度")
ax.set_xlabel("商品累計占比（%，依銷量由高到低排序）")
ax.set_ylabel("交易累計占比（%）")
ax.set_xlim(0, 100)
ax.set_ylim(0, 101)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**四條確認無誤**：

- **P50 = 8 個相異品項**——會議的數字正確，而且沒把「相異品項」跟「交易筆數」（中位數 9）搞混
- **2020 年 4 月實體店交易正好 0 筆**，同期線上創單月新高。這也是判定「通路 2 = 線上」的依據
- **季節性成立**：泳裝 6 月 11.5 萬筆 vs 1 月 2.4 萬筆，差 4.7 倍
- **長尾效應成立**：前 1% 品項佔 18.6% 交易、前 10% 佔 60.7%

以上全部推導完畢。下一格是最終規格。

---

# 建模共識

> **這頁是規格，不是資料摘要。**往下做召回、排序、對話層時照這裡寫的做；
> 資料本身長什麼樣請看 §1 附錄 A。
>
> 收錄標準：**說得出下游哪個模組會用到**。每一條都附理由，因為規格沒有理由就只會被當成建議。
> 所有數字由本 notebook 重跑產出。**與 09-23 會議記錄不同的地方集中在 §5。**

---

## 1. 時間切分

```python
訓練資料    t_dat <  '2020-09-16'
驗證週      2020-09-09 ~ 09-15     # 調參數只能看這裡
保留週      2020-09-16 ~ 09-22     # 只在最後跑一次
```

**為什麼要切三份，不是兩份**

「熱銷區間取幾天」「共購取前幾名」「分群取前幾名」都是**參數選擇**。做選擇就要看資料，
而**看了哪份資料，那份就不能再拿來誠實評估**——你等於用它的答案挑了一個對它有利的設定。

所以驗證週的存在是為了「做決定」，保留週的存在是為了「被評估」。
**任何人要調參數，一律在驗證週上調；保留週的分數在論文定稿前只跑一次。**

---

## 2. 評估規格

**適用於所有模型**——B0、B1、排序模型、加對話的版本，全部用同一把尺量。
下面每一條都是評估腳本的岔路口：同一份資料、同一個模型，寫得不一樣就會算出不可比的分數。

**答案卷（正解怎麼定）**

- **評估母體**：保留週有購買的 **68,984** 位顧客
- **正解**：該顧客保留週買過的**相異** `article_id`；

答案卷就是一張查詢表，一位顧客一列：

```
顧客 A  →  [X, Y]        2 組
顧客 B  →  [Y]           1 組
顧客 C  →  [X, Z, W]     3 組
```

評估時拿模型吐出的 12 個商品，去跟該顧客那一列對——中了幾個、中在第幾位，就是 AP 在算的東西。

全體共 **213,728** 組（顧客 × 品項）。這是**把每個人買的件數加總**。
它不參與計算，只作為小組檢查彼此結果之用：撈出來不是這個數字就是寫錯了，最常見是忘了去重
（同一件買兩次被算兩組，會變成 240,311）或日期邊界差一天。

**答案紙（模型要交什麼）**

- **長度 12，不得有重複品項。** 重複會讓 AP 算出大於 1 的分數，而且不會報錯
- **不足 12 件時以 B0 補滿**，空著的位置視為未命中
- **允許推薦顧客已買過的商品**——重購是 Stage 1 的召回源之一，先過濾掉等於自廢一個源

**怎麼打分**

- **主要指標 `MAP@12`**，AP 分母取 `min(正解數, 12)`。
  定義見《B0 基準線的單週評估風險與滾動驗證》附錄「MAP@12 定義」
- **次要指標 `Recall@12`** ＝ `命中數 / 正解數`，不套 `min()`——
  買 50 件的顧客用 12 個名額本來就不可能全中，這是指標的誠實上限

> ⚠️ 母體一定要註明。若改用全體 1,371,980 位顧客當分母，同一個模型的分數會變成
> **1/19.9**。提案 §3.2 對照的 Kaggle 銀牌 0.02996 是買家分母，兩者才可比。

---

## 3. 缺失值處理規格

`tx` 五個欄位**完全無缺失**，以下只有 `cus` 與 `art`。

| 表 | 欄位 | 缺失 | 怎麼做 | 為什麼 |
|---|---|---|---|---|
| `cus` | `FN` | 65.24% | **補 0** | 這欄**只出現過 `1.0`，資料裡沒有任何 `0`**。NULL 的語意是「沒有這個標記」，補 0 是還原它本來的意思 |
| `cus` | `Active` | 66.15% | **補 0** | 同上。另：`Active=1` 的人平均買 29.0 筆、NULL 的人 20.4 筆，補 0 後這個差異會被模型學到 |
| `cus` | `age` | 1.16% | **保留為 `未填` 類別，不刪顧客** | 這 15,861 人裡有 **292 位是保留週買家**。刪掉母體會從 68,984 變 68,692，分數與提案、與彼此都不可比。代價（1.16% 的人）遠小於失去可比性 |
| `cus` | `club_member_status` | 0.44% | **補 `UNKNOWN` 類別** | NULL 者平均買 10.3 筆 vs 有值者 23.2 筆 —— **NULL 本身帶訊息**，補眾數會把這個訊號抹掉 |
| `cus` | `fashion_news_frequency` | 1.17% | **補 `NONE`，並先合併髒值** | 資料裡同時有 `NONE`（877,711）與 `None`（2），是同一個意思的兩種寫法，不合併會多出一個幾乎空的類別 |
| `cus` | `postal_code` | 0% | **整欄不使用** | 352,899 個相異值、平均每區 3.9 人；且最大的一區有 120,303 人（8.77%）而第二名只有 261 人 —— **差 461 倍，那是佔位值不是真郵遞區號** |
| `art` | `detail_desc` | 0.39% | **以「名稱 + 類型 + 色系」組成替代文字，不刪商品** | 這 416 個商品裡有 **17 個是保留週的正解品項**，刪掉等於自己把召回上限往下砍。替代文字的理由見下方改動紀錄 |

**改動紀錄：`detail_desc`（2026-09-28）**

- **原本定的**：補空字串
- **改成**：以「名稱 + 類型 + 色系」組成替代文字，採用提案 §2.1.2 既有的做法
- **理由**：這 416 個商品是向量檢索（Stage 1 第四個召回源）的輸入。補空字串會讓它們的
  文字向量全部退化成同一個點，**等於永遠檢索不到**——本來只是「缺一欄」，結果變成
  「整件商品在那個召回源裡消失」。用名稱、商品類型、色系拼出來的文字雖然比原始描述粗，
  但至少保有可比對的語意。兩種做法都不刪商品，差別只在缺的那一欄拿什麼填。
- **附帶效果**：這條原本與提案 §2.1.2 的「以名稱 + 類型 + 色系組成替代文字」衝突，改完後兩份文件一致。

**判準是什麼**

會議當時的規則是「缺失多的補、缺失少的刪」。這條規則用**缺失比例**當判準，
但比例不告訴你真正該看的兩件事：

1. **NULL 的語意**。`FN` 補 0 是對的，但不是因為缺 65%，而是因為那欄沒有 `0` 這個值。
   如果它原本就有 0 和 1，65% 的 NULL 反而必須當第三類，補 0 會製造假資料。
2. **刪的是哪張表的列**。刪 `tx`（事實表）＝少幾筆交易。
   刪 `cus` / `art`（維度表）＝把顧客／商品整個移出宇宙，評估母體與候選池會跟著變。
   `age` 缺 1.16% 聽起來可忽略，但它動到的是評估母體。

**所以規格是：看語意 + 看表，不看比例。本專案不刪任何維度表的列。**

---

## 4. 基準線

| 基準線 | 定義 | MAP@12 | 命中率 |
|---|---|---|---|
| **B0** | 近 14 天全站熱銷前 12 | **0.00710** | 6.1% |
| **B1** | (年齡層 × 主要品牌線) 分群熱銷前 12，不足補 B0；35 個分群 | **0.00927** | 6.8% |

B1 相對 B0 **+30.5%**。提案目標 MAP@12 ≥ 0.025 ＝ **B1 的 2.7 倍**。

**為什麼熱銷區間是 14 天**

候選有 7 / 14 / 21 / 30 / 60 天與「保留週以外全部」。在**驗證週**上掃描，14 天最高，於是採用。
關鍵是這個決定完全沒有用到保留週。

```
熱銷區間          驗證週 MAP    保留週 MAP
近 7 天           0.00660      0.00875
近 14 天          0.00700      0.00710   ← 驗證週選出
近 21 天          0.00617      0.00604
近 30 天          0.00558      0.00548
保留週以外全部      0.00344      0.00290
```

兩個陷阱都要避開：

- **近 7 天**在保留週拿 0.00875 是六個候選裡最高的，但在驗證週只排第二。
  那個高分有一部分是那一週的運氣 —— **用保留週的分數挑參數，就是用測試集調參**。
- **「保留週以外全部」**零調參，但 MAP 只有 0.00290，比 14 天弱 2.4 倍。
  基準線太弱會讓模型的提升虛胖，這跟基準線太強是同一個錯誤的兩面。
  **基準線的職責是當一個公平而且盡力的對手**，不是當一個中性的參考點。

14 天在兩個獨立的週分別是 0.00700 與 0.00710，幾乎一樣；7 天是 0.00660 → 0.00875，跳 33%。
**穩定本身就是可信度的證據。**

> ⚠️ **可重現性**：`arg_max()` 與 `ORDER BY count(*) DESC` 在平手時**不具決定性**，
> 同一份資料連跑五次分群人數會跳動。**所有排序都要加決勝欄位**，
> 例如 `ORDER BY n DESC, article_id`。否則你分不出分數變動是模型改善還是雜訊。

---

## 5. 與 09-23 會議記錄不同的地方

每一條都附證據，要退回會議原案隨時可以。

| # | 會議記錄 | 改成 | 理由 |
|---|---|---|---|
| 1 | 熱銷區間近 30 天 | **近 14 天** | 30 天在驗證週 0.00558、14 天 0.00700。原本的理由「最近 30 天新商品被購買比例最高」實測不成立：最近 30 天首次上架的商品平均交易數 56.8，比更早上架的 321.9 **低** |
| 2 | B1 只用年齡分群 | **年齡 × 品牌線** | 提案 §2.2.4 的原始定義。同樣 14 天下，只用年齡 0.00888（7 群）、加品牌線 0.00927（35 群） |
| 3 | `age` 缺失刪除 | **保留為 `未填`** | 會刪掉 292 位保留週買家，評估母體從 68,984 變 68,692 |
| 4 | 缺失 <1–2% 的欄位刪列 | **維度表一律不刪列** | `detail_desc` 那 416 個商品含 17 個保留週正解 |
| 5 | 切分寫「9/15–9/22」（8 天） | **9/16–9/22（7 天）** | 9/15 有 26,053 筆交易。放錯邊保留週會變 266,364 筆，母體跟著變，與提案的 68,984 不可比 |
| 6 | 相鄰兩週熱門重合率 28% | **39–50%** | 實測：2020 全年 40.1%、最近 12 週 38.9%、最近 4 週 50.0%。28% 疑似與附錄 A 的「最近四週有成交品項佔目錄 26.8%」混淆 |

**會議記錄中經實測確認無誤的**：P50 = 8 個相異品項、疫情期間線上通路 100%（2020 年 4 月
實體店交易正好 0 筆）、季節性（泳裝 6 月 11.5 萬 vs 1 月 2.4 萬）、長尾效應、
`FN`/`Active` 補 0、B0 ≈ 0.005 與 B1 = 0.0069（在 30 天設定下完全重現）。